# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library. We'll access and analyze the FAIR² mass spectrometry dataset containing records about protein abundance, modifications, and related measurements from human mast cell extracellular vesicles.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"Published: {getattr(metadata, 'datePublished', None)}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

We inspect the record sets defined within the dataset. These are collections of related records (e.g., a table). For each record set, we list its `@id`, name, and the fields (columns) it includes (by their `@id`).

In [ ]:
# Extract all record set metadata
record_sets = getattr(metadata, 'recordSet', [])
if not hasattr(record_sets, '__iter__') or isinstance(record_sets, str):
    # If single, make it a list
    record_sets = [record_sets]

print(f"Number of record sets: {len(record_sets)}\n")

# We'll extract their @id, name, and fields
record_set_info = []

for rs in record_sets:
    name = getattr(rs, 'name', None)
    rs_id = getattr(rs, '@id', None)
    fields = getattr(rs, 'field', [])
    if not hasattr(fields, '__iter__') or isinstance(fields, str):
        fields = [fields]
    field_ids = [getattr(f, '@id', str(f)) for f in fields]
    print(f"RecordSet: {rs_id if rs_id else '<no @id>'} | Name: {name}")
    print(f"  Number of fields: {len(field_ids)}")
    print(f"  Field @ids: {field_ids}\n")
    record_set_info.append({'@id': rs_id, 'fields': field_ids, 'name': name})

if not record_sets:
    print("No explicit record sets found in metadata. Attempting fallback via .list_record_sets() API.")
    # Fallback: try listing record sets by API
    try:
        available_sets = dataset.list_record_sets()
        print("Available record sets by API:", available_sets)
    except Exception as e:
        print(f"Could not enumerate record sets: {e}")


## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. We use the record set and field `@id`s from the previous overview.

Most Croissant datasets have at least one central record set. We'll attempt to extract all data tables. Make sure to use the record set `@id` string when loading records.

In [ ]:
# If recordSet metadata provided the @ids, use them. If not, try list_record_sets API.
record_set_ids = [rsi['@id'] for rsi in record_set_info if rsi['@id']] if 'record_set_info' in locals() and record_set_info else []

if not record_set_ids:
    record_set_ids = dataset.list_record_sets()

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        # Save only if not empty
        if not df.empty:
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} rows for record set @id: {record_set_id}")

# Display available tables
print(f"\nAvailable DataFrames: {list(dataframes.keys())}")

# Show columns for one DataFrame as example
selected_rs = list(dataframes.keys())[0] if dataframes else None
if selected_rs:
    print(f"\nColumns in '{selected_rs}':", dataframes[selected_rs].columns.tolist())
    display(dataframes[selected_rs].head())
else:
    print("No dataframes extracted. Check schema or data availability.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select a numeric field from the extracted table and demonstrate filtering and normalization. **Remember:** reference fields by their exact `@id`. (You can print out all columns using `dataframes[record_set_id].columns`.)

In [ ]:
import numpy as np
# Set your record set and field @id here. Use the output from previous cell, e.g.:
record_set_id = selected_rs

# Inspect columns
print(f"Fields in the chosen DataFrame ({record_set_id}):\n{dataframes[record_set_id].columns.tolist()}")

# Pick a numeric column; If not known, print all and choose e.g. 'cr:abundance_sample1'
numeric_field_id = None
for col in dataframes[record_set_id].columns:
    # Heuristically find a likely numeric field
    if 'abundance' in col or 'count' in col or 'MW' in col or 'coverage' in col:
        numeric_field_id = col
        break
if not numeric_field_id:
    # If none matched, just use the first column
    numeric_field_id = dataframes[record_set_id].columns[0]
print(f"Using numeric field: {numeric_field_id}\n")

# Coerce to numeric, if not already
df = dataframes[record_set_id]
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
threshold = df[numeric_field_id].quantile(0.75)
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (upper quartile): {len(filtered_df)} records\n")

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by another field (if it exists)
group_field = None
for col in df.columns:
    if 'description' in col or 'type' in col or 'group' in col:
        group_field = col
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field, as_index=False)[numeric_field_id].mean()
    print(f"\nGrouped data by {group_field} (mean of {numeric_field_id}):\n")
    print(grouped_df.head())
else:
    print("No categorical field found for grouping.")

## 5. Visualization

Visualize data distributions and relationships in the dataset. For example, show the histogram of the chosen numeric field and a boxplot by group, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the normalized numeric field
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True, bins=20)
plt.title(f"Distribution of Normalized {numeric_field_id}")
plt.xlabel(f"Normalized {numeric_field_id}")
plt.ylabel("Count")
plt.show()

# Boxplot by group, if available
if group_field:
    plt.figure(figsize=(12,4))
    sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we loaded the FAIR² mass spectrometry dataset, explored its structure using `mlcroissant`, and performed basic data wrangling and visualization.

**Key findings:**
- Loaded and described available record sets and fields using their `@id`s.
- Extracted and filtered protein abundance data.
- Applied normalization and visualized distributions.

For deeper domain-specific insights, explore field definitions and annotations in the Croissant schema and apply domain knowledge to the protein analysis.
